# Phase 1 · Data Loading & Preprocessing (CIFAR-10)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MartinAlani/project_deep_learning_week_7/blob/main/notebooks/01_data_preprocessing.ipynb)

**Goals of this notebook**
1. Load the CIFAR-10 dataset.
2. Understand its structure (shapes, classes, class distribution).
3. Visualize images together with their labels.
4. Split the data into **train / validation / test** sets.
5. Normalize pixel values.
6. Set up **data augmentation** and an efficient `tf.data` input pipeline.

> 💡 **In Colab:** go to *Runtime → Change runtime type → T4 GPU* before running anything.

## 1. Imports and random seed

Fixing a **random seed** makes every random operation (data split, weight initialization, shuffling...) **reproducible**: running the notebook twice gives the same results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and TensorFlow at once

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

## 2. Load CIFAR-10

Keras ships CIFAR-10 as a built-in dataset; it is downloaded (~170 MB) the first time.

- `x_*` are the **images** (model inputs).
- `y_*` are the **labels** (an integer 0-9 identifying the class).

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Labels come with shape (N, 1); flatten them to (N,) for convenience
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print("Full train set:", x_train_full.shape, y_train_full.shape)
print("Test set:      ", x_test.shape, y_test.shape)
print("Data type:", x_train_full.dtype, "| Pixel range:", x_train_full.min(), "-", x_train_full.max())

**How to read the shape `(50000, 32, 32, 3)`:**
- `50000` → number of images
- `32, 32` → height and width in pixels
- `3` → colour channels (Red, Green, Blue)

Each pixel value is an integer between **0 and 255**.

## 3. Class distribution

We check whether the dataset is **balanced** (same number of images per class). If it were not, accuracy alone could be a misleading metric.

In [ ]:
classes, counts = np.unique(y_train_full, return_counts=True)

plt.figure(figsize=(10, 4))
plt.bar([CLASS_NAMES[c] for c in classes], counts, color="steelblue")
plt.title("Number of images per class (train set)")
plt.ylabel("Images")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

for c, n in zip(classes, counts):
    print(f"{CLASS_NAMES[c]:>11}: {n}")

✅ All 10 classes have exactly 5,000 images: the dataset is **perfectly balanced**, so accuracy is a reliable metric.

## 4. Visualize images and labels

In [ ]:
def show_images(images, labels, n_rows=4, n_cols=8, title=None):
    """Display a grid of images with their class name above each one.

    Args:
        images: array of images with shape (N, height, width, 3).
        labels: array of integer class labels (0-9).
        n_rows, n_cols: grid size.
        title: optional figure title.
    """
    plt.figure(figsize=(n_cols * 1.6, n_rows * 1.8))
    for i in range(n_rows * n_cols):
        plt.subplot(n_rows, n_cols, i + 1)
        plt.imshow(images[i])
        plt.title(CLASS_NAMES[labels[i]], fontsize=9)
        plt.axis("off")
    if title:
        plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# 32 randomly chosen images
idx = np.random.choice(len(x_train_full), 32, replace=False)
show_images(x_train_full[idx], y_train_full[idx], title="Random CIFAR-10 samples")

Now a few examples of **each class**, to get a feeling for what the model has to learn:

In [ ]:
plt.figure(figsize=(15, 8))
for class_id in range(10):
    class_idx = np.where(y_train_full == class_id)[0][:6]   # first 6 images of this class
    for j, i in enumerate(class_idx):
        plt.subplot(6, 10, j * 10 + class_id + 1)
        plt.imshow(x_train_full[i])
        plt.axis("off")
        if j == 0:
            plt.title(CLASS_NAMES[class_id], fontsize=10)
plt.suptitle("6 examples per class", fontsize=14)
plt.tight_layout()
plt.show()

🔎 **Observation:** images are very small (32×32) and some classes look alike (**cat** vs **dog**, **automobile** vs **truck**). We expect the model to confuse these pairs; we will check it with the confusion matrix in Phase 4.

## 5. Train / validation / test split

| Set | Purpose | Size |
|---|---|---|
| **Train** | The model learns from this data | 45,000 |
| **Validation** | Monitor training (early stopping, hyper-parameter choices) | 5,000 |
| **Test** | **Final** evaluation, used only once at the end | 10,000 |

⚠️ **Common mistake:** using the test set to make decisions during training. This *leaks* information and gives an over-optimistic result.

`stratify` keeps the same class proportions in every split.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=5000,
    stratify=y_train_full,
    random_state=SEED,
)

print("Train:     ", x_train.shape)
print("Validation:", x_val.shape)
print("Test:      ", x_test.shape)
print("Images per class in validation:", np.bincount(y_val))

## 6. Normalization

Neural networks train better with small input values. We rescale pixels from **[0, 255]** to **[0, 1]** by dividing by 255.

> 📝 We will apply normalization **inside the model** (as a `Rescaling` layer). That way, at deployment time (Phase 6), we can feed raw images and the model normalizes them itself. This prevents a very common production bug: forgetting to normalize.

Here we just check that it works:

In [ ]:
normalize = layers.Rescaling(1.0 / 255)

sample = normalize(x_train[:5].astype("float32"))
print("Before: min =", x_train[:5].min(), "| max =", x_train[:5].max())
print("After:  min =", float(tf.reduce_min(sample)), "| max =", float(tf.reduce_max(sample)))

## 7. Data augmentation

**Data augmentation** creates random variations of the training images (flips, small shifts, zoom...). The model sees slightly "new" images every epoch, which helps it **generalize** and reduces **overfitting**.

We only use **realistic** transformations: a horizontally flipped ship is still a ship, but an upside-down ship (vertical flip) is not something we expect to see.

⚠️ Augmentation is applied **only during training**, never to validation or test data. Keras augmentation layers handle this automatically: they are active only when `training=True`.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),         # left-right flip
    layers.RandomTranslation(0.1, 0.1),      # shift up to 10%
    layers.RandomZoom(0.1),                  # zoom up to 10%
], name="data_augmentation")

In [ ]:
# Show 8 augmented versions of the same image
image = x_train[7]

plt.figure(figsize=(14, 2.4))
plt.subplot(1, 9, 1)
plt.imshow(image)
plt.title("Original", fontsize=9)
plt.axis("off")
for i in range(8):
    augmented = data_augmentation(image[np.newaxis].astype("float32"), training=True)
    plt.subplot(1, 9, i + 2)
    plt.imshow(augmented[0].numpy().astype("uint8"))
    plt.title(f"Augmented {i+1}", fontsize=9)
    plt.axis("off")
plt.suptitle(f"Data augmentation applied to a '{CLASS_NAMES[y_train[7]]}'")
plt.tight_layout()
plt.show()

## 8. Efficient input pipeline with `tf.data`

`tf.data` feeds the data in **batches** and prepares the next batch while the GPU is busy with the current one (`prefetch`), which speeds up training.

- `shuffle` → reshuffles the training data every epoch (so the model cannot learn the order).
- `batch` → groups images 64 at a time.
- `prefetch` → prepares batches ahead of time.

In [ ]:
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(x, y, training=False):
    """Turn NumPy arrays into a batched, prefetched tf.data.Dataset.

    Args:
        x: images (N, 32, 32, 3) as uint8.
        y: integer labels (N,).
        training: if True, shuffle the data (train set only).
    """
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(buffer_size=len(x), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val)
test_ds = make_dataset(x_test, y_test)

# Inspect one batch
imgs, labels = next(iter(train_ds))
print("Image batch shape:", imgs.shape)
print("Label batch shape:", labels.shape)
print("First labels:", [CLASS_NAMES[i] for i in labels[:5].numpy()])

## ✅ Phase 1 summary

| Step | Decision | Why |
|---|---|---|
| Split | 45k train / 5k val / 10k test, stratified | Validation drives early stopping; the test set stays untouched for the final evaluation |
| Normalization | `Rescaling(1/255)` **inside the model** | Inputs in [0,1] → stable training; simpler deployment |
| Augmentation | Horizontal flip, 10% shift, 10% zoom | Reduces overfitting; only realistic transformations |
| Pipeline | `tf.data` with shuffle, batch 64, prefetch | Faster training |

👉 **For the report:** save the plots from this notebook (right-click → *Save image as*); they are useful for the preprocessing section.

**Next step → Phase 2:** design our own CNN.